# Modelo Bayesian: Naive Bayes

Notebook elaborado siguiendo la rúbrica de la asignatura (ver `../../rubrica.md`).

**Autor:** Grupo E
**Fecha:** 13/06/2026

## 1. Descripción

**Naive Bayes** es una familia de clasificadores bayesianos supervisados basada en el teorema de Bayes y en la suposición de independencia condicional entre atributos dado la clase. Su objetivo es estimar la clase más probable para una instancia a partir de probabilidades posteriores.

Dada una instancia $X = (x_1, x_2, \\dots, x_m)$, el clasificador calcula $P(C\mid X)$ y asigna la clase con mayor valor posterior. Aunque la independencia condicional es una suposición fuerte, el método suele funcionar muy bien en muchos problemas reales.

### Casos de uso típicos
- Clasificación de correos o spam.
- Clasificación de texto y sentimiento.
- Diagnóstico con variables discretas.
- Clasificación de riesgo en datos tabulares.
- Filtrado y categorización de documentos.

## 2. Bibtex y Referencias

### BibTeX
```bibtex
@book{mitchell1997machine,
  title     = {Machine Learning},
  author    = {Mitchell, Tom M.},
  publisher = {McGraw-Hill},
  year      = {1997}
}

@article{domingos1997optimality,
  title   = {On the optimality of the simple Bayesian classifier under zero-one loss},
  author  = {Domingos, Pedro and Pazzani, Michael},
  journal = {Machine Learning},
  volume  = {29},
  number  = {2-3},
  pages   = {103--130},
  year    = {1997}
}
```

### APA
- Mitchell, T. M. (1997). *Machine Learning*. McGraw-Hill.
- Domingos, P., & Pazzani, M. (1997). *On the optimality of the simple Bayesian classifier under zero-one loss*. **Machine Learning, 29**(2-3), 103--130.

## 3. Tipo de Modelo

| Criterio | Clasificación |
|---|---|
| **Método de aprendizaje** | Supervisado |
| **Por parámetros** | Paramétrico |
| **Datos de aprendizaje** | Offline (batch) |
| **Resultado del entrenamiento** | Clasificador probabilístico bayesiano |

Notas:
- **Supervisado** porque aprende con etiquetas de clase.
- **Paramétrico** porque estima un conjunto finito de probabilidades.
- **Offline** porque requiere un conjunto de entrenamiento completo.
- El resultado es un modelo de decisión probabilística interpretable.

## 4. Algoritmo de Entrenamiento

Naive Bayes estima la clase más probable aplicando el teorema de Bayes y asumiendo independencia condicional entre atributos.

### Pseudocódigo
```text
Entrada: D = {(x^(i), c^(i))}
Salida: clasificador bayesiano

1. Calcular P(C) para cada clase.
2. Calcular P(x_j | C) para cada atributo j y clase C.
3. Para una nueva instancia x, calcular:
   P(C | x) ∝ P(C) ∏ P(x_j | C)
4. Elegir la clase con mayor probabilidad posterior.
```

### Métricas clave

- **Probabilidad posterior:** $P(C \mid X)$
- **Exactitud:** proporción de predicciones correctas
- **Precisión:** proporción de positivos predichos correctamente
- **Recall:** proporción de positivos reales recuperados
- **F1-score:** equilibrio entre precisión y recall

## 5. Supuestos y Restricciones

- **Independencia condicional aproximada:** los atributos se tratan como independientes dado la clase.
- **Atributos discretos o categóricos:** si hay variables continuas, deben discretizarse o usar otra variante.
- **Etiquetas disponibles:** el aprendizaje requiere ejemplos etiquetados.
- **Suavizado recomendado:** evita probabilidades nulas.
- **Sensibilidad a datos escasos:** con muy pocas observaciones, las frecuencias pueden ser inestables.
- **Variables faltantes:** deben tratarse antes del entrenamiento.

## 6. Tests / Métricas de validación

Naive Bayes se valida como clasificador mediante métricas de desempeño predictivo:

- **Exactitud (accuracy)** — porcentaje total de aciertos.
- **Precisión (precision)** — calidad de las predicciones positivas.
- **Recall** — cobertura de la clase positiva.
- **F1-score** — balance entre precisión y recall.
- **Matriz de confusión** — resumen de aciertos y errores.
- **Probabilidades posteriores** — interpretación de la confianza de decisión.

---
## 7. Implementación práctica

Usaremos un ejemplo categórico pequeño para ilustrar Naive Bayes desde cero.

### 7.1 Instalación e imports

In [2]:
import pandas as pd
import numpy as np

print('Librerías cargadas correctamente')

Librerías cargadas correctamente


### 7.2 Dataset de ejemplo

Usamos el clásico conjunto de clima y decisión para clasificación binaria.

In [3]:
datos = pd.DataFrame([
    ['soleado', 'caliente', 'alta', 'falso', 'no'],
    ['soleado', 'caliente', 'alta', 'verdadero', 'no'],
    ['nublado', 'caliente', 'alta', 'falso', 'si'],
    ['lluvioso', 'templado', 'alta', 'falso', 'si'],
    ['lluvioso', 'frio', 'normal', 'falso', 'si'],
    ['lluvioso', 'frio', 'normal', 'verdadero', 'no'],
    ['nublado', 'frio', 'normal', 'verdadero', 'si'],
    ['soleado', 'templado', 'alta', 'falso', 'no'],
    ['soleado', 'frio', 'normal', 'falso', 'si'],
    ['lluvioso', 'templado', 'normal', 'falso', 'si'],
    ['soleado', 'templado', 'normal', 'verdadero', 'si'],
    ['nublado', 'templado', 'alta', 'verdadero', 'si'],
    ['nublado', 'caliente', 'normal', 'falso', 'si'],
    ['lluvioso', 'templado', 'alta', 'verdadero', 'no'],
], columns=['cielo', 'temperatura', 'humedad', 'viento', 'jugar'])

datos

,cielo,temperatura,humedad,viento,jugar
0,soleado,caliente,alta,falso,no
1,soleado,caliente,alta,verdadero,no
2,nublado,caliente,alta,falso,si
3,lluvioso,templado,alta,falso,si
4,lluvioso,frio,normal,falso,si
5,lluvioso,frio,normal,verdadero,no
6,nublado,frio,normal,verdadero,si
7,soleado,templado,alta,falso,no
8,soleado,frio,normal,falso,si
9,lluvioso,templado,normal,falso,si


### 7.3 Entrenamiento y prueba

Separaremos una parte de los datos para evaluar el clasificador.

In [4]:
def dividir_datos(df, proporcion_prueba=0.3, semilla=42):
    rng = np.random.default_rng(semilla)
    indices = rng.permutation(len(df))
    n_prueba = max(1, int(len(df) * proporcion_prueba))
    idx_prueba = indices[:n_prueba]
    idx_entrenamiento = indices[n_prueba:]
    return df.iloc[idx_entrenamiento].reset_index(drop=True), df.iloc[idx_prueba].reset_index(drop=True)

train_df, test_df = dividir_datos(datos, proporcion_prueba=0.3, semilla=42)

print(f'Tamaño entrenamiento: {len(train_df)}')
print(f'Tamaño prueba: {len(test_df)}')
train_df

Tamaño entrenamiento: 10
Tamaño prueba: 4


,cielo,temperatura,humedad,viento,jugar
0,soleado,caliente,alta,falso,no
1,nublado,templado,alta,verdadero,si
2,lluvioso,templado,alta,verdadero,no
3,soleado,templado,normal,verdadero,si
4,lluvioso,frio,normal,verdadero,no
5,nublado,caliente,alta,falso,si
6,lluvioso,frio,normal,falso,si
7,nublado,caliente,normal,falso,si
8,soleado,caliente,alta,verdadero,no
9,soleado,frio,normal,falso,si


### 7.4 Implementación de Naive Bayes

In [5]:
class NaiveBayesClasificador:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.target_col = None
        self.feature_cols = None
        self.train_df = None
        self.class_values = None
        self.feature_values = None

    def fit(self, df, target_col):
        self.train_df = df.copy()
        self.target_col = target_col
        self.feature_cols = [c for c in df.columns if c != target_col]
        self.class_values = sorted(df[target_col].unique())
        self.feature_values = {c: sorted(df[c].dropna().unique()) for c in self.feature_cols}
        return self

    def _laplace(self, count, total, cardinality):
        return (count + self.alpha) / (total + self.alpha * cardinality)

    def _count(self, conditions):
        mask = pd.Series(True, index=self.train_df.index)
        for col, value in conditions.items():
            mask &= self.train_df[col].eq(value)
        return int(mask.sum())

    def predict_proba_one(self, row):
        scores = {}
        total_rows = len(self.train_df)
        for class_value in self.class_values:
            class_count = self._count({self.target_col: class_value})
            score = self._laplace(class_count, total_rows, len(self.class_values))
            for feature in self.feature_cols:
                feature_count = self._count({self.target_col: class_value, feature: row[feature]})
                score *= self._laplace(feature_count, class_count, len(self.feature_values[feature]))
            scores[class_value] = score
        total = sum(scores.values())
        return {c: s / total for c, s in scores.items()} if total else {c: 1 / len(scores) for c in scores}

    def predict_one(self, row):
        proba = self.predict_proba_one(row)
        return max(proba, key=proba.get)

    def predict(self, df):
        return df.apply(self.predict_one, axis=1)

modelo = NaiveBayesClasificador(alpha=1.0)
modelo.fit(train_df, target_col='jugar')
print('Modelo Naive Bayes entrenado correctamente')

Modelo Naive Bayes entrenado correctamente


### 7.5 Predicción sobre el conjunto de prueba

In [6]:
resultado = test_df.copy()
resultado['prediccion'] = modelo.predict(test_df)
resultado['acierto'] = resultado['jugar'] == resultado['prediccion']
resultado

,cielo,temperatura,humedad,viento,jugar,prediccion,acierto
0,nublado,frio,normal,verdadero,si,si,True
1,soleado,templado,alta,falso,no,si,False
2,lluvioso,templado,normal,falso,si,si,True
3,lluvioso,templado,alta,falso,si,no,False


### 7.6 Métricas de evaluación

In [7]:
def matriz_confusion_binaria(y_real, y_pred, clase_positiva='si'):
    tp = int(((y_real == clase_positiva) & (y_pred == clase_positiva)).sum())
    tn = int(((y_real != clase_positiva) & (y_pred != clase_positiva)).sum())
    fp = int(((y_real != clase_positiva) & (y_pred == clase_positiva)).sum())
    fn = int(((y_real == clase_positiva) & (y_pred != clase_positiva)).sum())
    return tp, tn, fp, fn

def metricas_binarias(y_real, y_pred, clase_positiva='si'):
    tp, tn, fp, fn = matriz_confusion_binaria(y_real, y_pred, clase_positiva)
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

metricas = metricas_binarias(resultado['jugar'], resultado['prediccion'], clase_positiva='si')
print(f"Exactitud: {metricas['accuracy']:.3f}")
print(f"Precisión: {metricas['precision']:.3f}")
print(f"Recall: {metricas['recall']:.3f}")
print(f"F1-score: {metricas['f1']:.3f}")

cm = pd.DataFrame([[metricas['tn'], metricas['fp']], [metricas['fn'], metricas['tp']]], index=['Real: no', 'Real: si'], columns=['Pred: no', 'Pred: si'])
cm

Exactitud: 0.500
Precisión: 0.667
Recall: 0.667
F1-score: 0.667


,Pred: no,Pred: si
Real: no,0,1
Real: si,1,2


### 7.7 Resultados del modelo

Mostramos la predicción y la probabilidad posterior para cada instancia de prueba.

In [8]:
for idx, fila in test_df.iterrows():
    proba = modelo.predict_proba_one(fila)
    pred = max(proba, key=proba.get)
    print(f"Instancia {idx + 1} -> clase real: {fila['jugar']} | predicción: {pred} | probabilidades: {proba}")

Instancia 1 -> clase real: si | predicción: si | probabilidades: {'no': 0.15724815724815722, 'si': 0.8427518427518428}
Instancia 2 -> clase real: no | predicción: si | probabilidades: {'no': 0.42737896494156924, 'si': 0.5726210350584306}
Instancia 3 -> clase real: si | predicción: si | probabilidades: {'no': 0.25141859450021825, 'si': 0.7485814054997818}
Instancia 4 -> clase real: si | predicción: no | probabilidades: {'no': 0.5281980742778541, 'si': 0.4718019257221458}


### 7.8 Traducción de resultados

Interpretación de los resultados obtenidos:

- El modelo estimó la clase más probable usando el teorema de Bayes.
- La predicción final corresponde a la clase con mayor probabilidad posterior.
- La matriz de confusión resume los aciertos y errores del clasificador.
- La exactitud indica el porcentaje global de predicciones correctas.
- Las probabilidades posteriores permiten interpretar el nivel de confianza del modelo.

## 8. Conclusión

- Naive Bayes es un clasificador bayesiano **supervisado** ampliamente usado por su simplicidad e interpretabilidad.
- Su hipótesis de independencia condicional reduce la complejidad del cálculo.
- El suavizado mejora la robustez frente a probabilidades nulas.
- Es especialmente útil en problemas con atributos discretos y alta dimensionalidad.
- En la práctica, sigue siendo una base sólida para clasificación probabilística.